# 008 向量内积与几何

先手算二维u=(2,1)、v=(1,3)。一条向量用一维数组，所有接口检查shape。

In [1]:
from pathlib import Path
import numpy as np
import experiment as lab
from test_experiment import run_tests
u = np.array([2.0, 1.0])
v = np.array([1.0, 3.0])
print("分量乘积", u * v, "内积", lab.dot(u, v))
print("长度", lab.norm2(u), lab.norm2(v), "距离", lab.distance(u, v))

分量乘积 [2. 3.] 内积 5.0
长度 2.23606797749979 3.1622776601683795 距离 2.23606797749979


## 投影与垂直残差
系数0.5，投影(0.5,1.5)，残差(1.5,-0.5)。

In [2]:
coefficient, projection, residual = lab.project_to_direction(u, v)
print(coefficient, projection, residual)
print("残差内积", lab.dot(residual, v))
print("平方长度", lab.dot(u, u), lab.dot(projection, projection) + lab.dot(residual, residual))
assert lab.dot(residual, v) == 0

0.5 [0.5 1.5] [ 1.5 -0.5]
残差内积 0.0
平方长度 5.0 5.0


## 同向不等于相同
余弦1，距离仍然不为0。

In [3]:
print("u与v", lab.cosine(u, v), lab.angle_degrees(u, v))
print("u与2u", lab.cosine(u, 2 * u), lab.distance(u, 2 * u))
assert abs(lab.angle_degrees(u, v) - 45) < 1e-12

u与v 0.7071067811865475 45.00000000000001
u与2u 0.9999999999999998 2.23606797749979


## 三维分数例子
内积-1，投影系数-1/13。

In [4]:
a = np.array([1.0, 2.0, -1.0])
b = np.array([2.0, 0.0, 3.0])
print(lab.describe_pair(a, b))
np.testing.assert_allclose(lab.project_to_direction(a, b)[1], [-2/13, 0, -3/13], rtol=0, atol=1e-12)

{'dimension': 3, 'u': [1.0, 2.0, -1.0], 'v': [2.0, 0.0, 3.0], 'elementwise_product': [2.0, 0.0, -3.0], 'sum': [3.0, 2.0, 2.0], 'difference': [-1.0, 2.0, -4.0], 'dot': -1.0, 'u_norm': 2.449489742783178, 'v_norm': 3.605551275463989, 'distance': 4.58257569495584, 'cosine': -0.11322770341445959, 'angle_degrees': 96.50141223339973, 'projection_coefficient': -0.07692307692307693, 'projection': [-0.15384615384615385, -0.0, -0.23076923076923078], 'residual': [1.1538461538461537, 2.0, -0.7692307692307692], 'residual_dot_direction': 0.0, 'squared_length_decomposition': [6.0, 0.07692307692307693, 5.9230769230769225]}


## 零向量与尺度反例
只捕捉预期定义域错误；不靠epsilon伪装定义。

In [5]:
zero = np.zeros(2)
print("零内积和范数", lab.dot(zero, u), lab.norm2(zero))
try:
    lab.cosine(zero, u)
except ValueError:
    print("零向量的余弦未定义，已拒绝")
s = np.array([1.0, 1.0])
t = np.array([1.0, -1.0])
scale = np.array([1.0, 2.0])
print("单轴缩放前后", lab.cosine(s, t), lab.cosine(s * scale, t * scale))

零内积和范数 0.0 0.0
零向量的余弦未定义，已拒绝
单轴缩放前后 0.0 -0.5999999999999999


## 100维的坐标检查
只显示前两维，会让98个不同向量都变为二维零向量。

In [6]:
basis = lab.standard_basis(100)
print("shape", basis.shape)
print("不同基向量距离", lab.distance(basis[0], basis[99]))
print("只看前两坐标的零行数", int(np.sum(np.all(basis[:, :2] == 0, axis=1))))
np.testing.assert_array_equal(basis @ basis.T, np.eye(100))

shape (100, 100)
不同基向量距离 1.4142135623730951
只看前两坐标的零行数 98


## 全套核验与保存
测试包含独立Fraction投影坐标，不以图形代替数值检查。

In [7]:
print(run_tests())
report = lab.run(Path.cwd() / "notebook_outputs")
print(report["invariances_and_counterexamples"])

{'status': 'passed', 'test_groups': ['complete hand-calculated 2D dot/norm/distance/angle/projection', '3D negative projection coefficient and exact rational coordinates', '100 independent Fraction dot/projection fixtures and nearest-line checks', 'same direction differs in distance; scaling and translation counterexamples', 'zero-vector distinctions, shape mismatch and nonfinite failures', 'projection coefficient overflow and post-conversion nonfinite values rejected', '100-dimensional coordinates and information loss in 2D display', 'repeated experiment produces identical outputs']}
{'same_direction_cosine': 0.9999999999999998, 'same_direction_distance': 2.23606797749979, 'original_pair_cosine': 0.7071067811865475, 'scaled_u_cosine': 0.7071067811865475, 'negative_u_cosine': -0.7071067811865475, 'translated_distance': 2.23606797749979, 'translated_cosine': 0.0, 'axis_scale_before_cosine': 0.0, 'axis_scale_after_cosine': -0.5999999999999999}


## 重启内核并全部执行
记录一个方向相同但距离不同的例子、一个尺度改变相似度的例子，以及二维展示不能支持的高维结论。